# Part 4: Model Training and Callbacks

In this module, we will train the model using our dynamically augmented dataset. 
We utilize Keras callbacks to ensure the model trains efficiently and automatically saves the best weights.

In [ ]:
import os
import cv2
import json
import shutil
import random
import datetime
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

In [ ]:
def load_root_env(env_name='.env'):
    """Load simple KEY=VALUE entries from the repo root .env without overwriting existing env vars."""
    start = Path.cwd().resolve()
    for folder in (start, *start.parents):
        env_path = folder / env_name
        if env_path.is_file():
            for line in env_path.read_text(encoding='utf-8').splitlines():
                line = line.strip()
                if not line or line.startswith('#') or '=' not in line:
                    continue
                key, value = line.split('=', 1)
                os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))
            print('Loaded environment from:', env_path)
            return env_path
    print('No root .env found; using existing environment and Colab secrets only.')
    return None

def get_hf_token():
    token = os.environ.get('HF_TOKEN')
    if token:
        return token
    try:
        from google.colab import userdata
        token = userdata.get('HF_TOKEN')
        if token:
            os.environ['HF_TOKEN'] = token
            return token
    except (ModuleNotFoundError, ImportError, KeyError):
        pass
    except Exception:
        pass
    return None

load_root_env()
HF_TOKEN = get_hf_token()

# Training writes to local models/. Git ignores this directory; Hugging Face Model is the source of truth.
PROJECT_ROOT = Path(os.environ.get("LIPY_PROJECT_ROOT", Path.cwd())).resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif PROJECT_ROOT.name == "training":
    PROJECT_ROOT = PROJECT_ROOT.parent

MODEL_DIR = Path(os.environ.get("LIPY_MODEL_DIR", PROJECT_ROOT / "models")).resolve()
MODEL_DIR.mkdir(parents=True, exist_ok=True)

DRIVE_MODEL_DIR = None
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_MODEL_DIR = Path('/content/drive/MyDrive/lipy/models')
    print('Google Drive model backup path:', DRIVE_MODEL_DIR)
except ModuleNotFoundError:
    print('Not running in Colab; skipping Google Drive mount.')

checkpoint_path = MODEL_DIR / "model.keras"
labels_path = MODEL_DIR / "labels.json"
history_path = MODEL_DIR / "training_history.json"
temp_model_dir = MODEL_DIR / ".training_tmp"
temp_model_dir.mkdir(parents=True, exist_ok=True)
temp_checkpoint_path = temp_model_dir / "best_model.keras"

# labels.json is kept next to model.keras so the backend can map class ids to Odia characters.
try:
    from backend.labels import odia_ml_labels
except ImportError:
    odia_ml_labels = {}
label_to_char = {label: char for char, label in odia_ml_labels.items()}
labels_data = [{"id": class_name, "char": label_to_char.get(class_name, "")} for class_name in valid_classes]
labels_json = json.dumps(labels_data, ensure_ascii=False, indent=2)

callbacks = [
    EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6),
    ModelCheckpoint(str(temp_checkpoint_path), monitor='val_accuracy', save_best_only=True, mode='max')
]

print("Temporary model checkpoint:", temp_checkpoint_path)
print("Final local model path:", checkpoint_path)
print("Final local labels path:", labels_path)


In [ ]:
history = model.fit(
    datagen.flow(X_train, y_train, batch_size=32),
    epochs=100,
    validation_data=(X_test, y_test),
    callbacks=callbacks,
    class_weight=class_weight_dict
)

if not temp_checkpoint_path.is_file():
    model.save(str(temp_checkpoint_path))

history_json = json.dumps(history.history, indent=2)
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
drive_artifact_paths = []

if DRIVE_MODEL_DIR is not None:
    DRIVE_MODEL_DIR.mkdir(parents=True, exist_ok=True)
    drive_checkpoint_path = DRIVE_MODEL_DIR / f"model_{timestamp}.keras"
    drive_labels_path = DRIVE_MODEL_DIR / f"labels_{timestamp}.json"
    drive_history_path = DRIVE_MODEL_DIR / f"training_history_{timestamp}.json"
    shutil.copyfile(temp_checkpoint_path, drive_checkpoint_path)
    drive_labels_path.write_text(labels_json, encoding="utf-8")
    drive_history_path.write_text(history_json, encoding="utf-8")
    drive_artifact_paths = [drive_checkpoint_path, drive_labels_path, drive_history_path]

shutil.copyfile(temp_checkpoint_path, checkpoint_path)
labels_path.write_text(labels_json, encoding="utf-8")
history_path.write_text(history_json, encoding="utf-8")

print("Training complete.")
if drive_artifact_paths:
    print("Timestamped Google Drive backup saved first:")
    for path in drive_artifact_paths:
        print(" -", path)
print("Local model artifacts saved to:")
print(" -", checkpoint_path)
print(" -", labels_path)
print(" -", history_path)


In [ ]:
print("Local model artifacts are ready:")
print(" -", checkpoint_path)
print(" -", labels_path)
print(" -", history_path)
if drive_artifact_paths:
    print("Google Drive backup artifacts:")
    for path in drive_artifact_paths:
        print(" -", path)
print("Upload local models/ with `python scripts/model/upload_hf.py` from the repo root.")
